In [1]:
## Important 

# B1 assumes A1 has been run at least once.

In [2]:
## Setup 

import sys
from pathlib import Path

def find_src_dir(start: Path = None) -> Path:
    current = start or Path.cwd()
    for _ in range(5):
        candidate = current / "src"
        if candidate.exists():
            return candidate
        current = current.parent
    raise FileNotFoundError("Could not locate a 'src' folder above the current directory.")

SRC_DIR = find_src_dir()
sys.path.append(str(SRC_DIR))

import polars as pl
from paths import DATA_DIR, IBES_DIR

In [3]:
## Volume profile: retail activity by trading day relative to announcement

# Used to select the event window empirically rather than by convention.
# Result: flat until -2, spike at day 0/+1 (~2.5x baseline), decay by +5.

from analysis.event_window_profile import build_event_profile

profile = build_event_profile(window=30)
with pl.Config(tbl_rows=-1):
    print(profile)

Matched 97,270 of 163,010 events to CBOE trading data
shape: (61, 3)
┌─────────┬─────────────────┬──────────┐
│ rel_day ┆ mean_retail_vol ┆ n_events │
│ ---     ┆ ---             ┆ ---      │
│ i64     ┆ f64             ┆ u32      │
╞═════════╪═════════════════╪══════════╡
│ -30     ┆ 705.993012      ┆ 94024    │
│ -29     ┆ 733.257719      ┆ 94122    │
│ -28     ┆ 730.185958      ┆ 94220    │
│ -27     ┆ 732.231435      ┆ 94342    │
│ -26     ┆ 722.294777      ┆ 94468    │
│ -25     ┆ 706.142591      ┆ 94606    │
│ -24     ┆ 720.85865       ┆ 94715    │
│ -23     ┆ 718.836406      ┆ 94820    │
│ -22     ┆ 706.241305      ┆ 94967    │
│ -21     ┆ 699.715598      ┆ 95126    │
│ -20     ┆ 681.803054      ┆ 95280    │
│ -19     ┆ 688.383603      ┆ 95385    │
│ -18     ┆ 694.000325      ┆ 95489    │
│ -17     ┆ 690.638102      ┆ 95654    │
│ -16     ┆ 680.298978      ┆ 95803    │
│ -15     ┆ 663.394114      ┆ 95962    │
│ -14     ┆ 681.641999      ┆ 96092    │
│ -13     ┆ 701.835271      ┆

In [4]:
## Sample representativness check

# Confirms the matched sample isn't concentrated in the 2020-21 retail boom.

import random

events = pl.read_parquet(IBES_DIR / "dispersion_events.parquet")
random.seed(42)
n_sample = 1000
if events.height > n_sample:
    idx = random.sample(range(events.height), n_sample)
    events_sample = events[idx]

year_counts = (
    events_sample.with_columns(pl.col("ANNDATS_ACT").dt.year().alias("year"))
    .group_by("year").len().sort("year")
)
with pl.Config(tbl_rows=-1):
    print(year_counts)

shape: (12, 2)
┌──────┬─────┐
│ year ┆ len │
│ ---  ┆ --- │
│ i32  ┆ u32 │
╞══════╪═════╡
│ 2011 ┆ 65  │
│ 2012 ┆ 81  │
│ 2013 ┆ 82  │
│ 2014 ┆ 103 │
│ 2015 ┆ 80  │
│ 2016 ┆ 97  │
│ 2017 ┆ 86  │
│ 2018 ┆ 81  │
│ 2019 ┆ 95  │
│ 2020 ┆ 82  │
│ 2021 ┆ 84  │
│ 2022 ┆ 64  │
└──────┴─────┘


In [5]:
## Composition comparison: retail vs. procust, near event vs. baseline

from analysis.event_window_profile import build_composition_comparison

comparison = build_composition_comparison()
comparison

Matched 97,270 of 163,010 events to CBOE trading data


participant_group,is_near_event,n_obs,share_lt_100,share_100_199,share_gt_199,share_call,share_put
str,bool,u32,f64,f64,f64,f64,f64
"""procust""",false,5230527,0.628166,0.057658,0.314175,0.58493,0.41507
"""procust""",true,581798,0.654998,0.063086,0.281915,0.574271,0.425729
"""retail""",false,5230527,0.705345,0.090275,0.20438,0.646288,0.353712
"""retail""",true,581798,0.719527,0.087516,0.192957,0.624221,0.375779


In [6]:
## Headline results: difference in differences across all outcomes

# treat:post is the answer to "does retail shift differently from professional customers", not merely "does retail shift".

from analysis.event_window_profile import build_diff_in_diff_panel, run_diff_in_diff

outcomes = ["lt_100", "call", "otm", "otm_call", "otm_put", "itm", "open"]

results = []
for oc in outcomes:
    panel = build_diff_in_diff_panel(outcome=oc)
    m_ev = run_diff_in_diff(panel, cluster_by="event")
    m_tk = run_diff_in_diff(panel, cluster_by="ticker")
    results.append({
        "outcome": oc,
        "did_coef": m_ev.params["treat:post"],
        "p_event_clustered": m_ev.pvalues["treat:post"],
        "p_ticker_clustered": m_tk.pvalues["treat:post"],
        "n_obs": int(m_ev.nobs),
    })

results_df = pl.DataFrame(results)
with pl.Config(tbl_rows=-1):
    print(results_df)

Matched 97,270 of 163,010 events to CBOE trading data
Panel has 317,190 rows (up to 2 periods x 2 groups per event)
Matched 97,270 of 163,010 events to CBOE trading data
Panel has 317,190 rows (up to 2 periods x 2 groups per event)
Matched 97,270 of 163,010 events to CBOE trading data
Panel has 317,190 rows (up to 2 periods x 2 groups per event)
Matched 97,270 of 163,010 events to CBOE trading data
Panel has 317,190 rows (up to 2 periods x 2 groups per event)
Matched 97,270 of 163,010 events to CBOE trading data
Panel has 317,190 rows (up to 2 periods x 2 groups per event)
Matched 97,270 of 163,010 events to CBOE trading data
Panel has 317,190 rows (up to 2 periods x 2 groups per event)
Matched 97,270 of 163,010 events to CBOE trading data
Panel has 317,190 rows (up to 2 periods x 2 groups per event)
shape: (7, 5)
┌──────────┬───────────┬───────────────────┬────────────────────┬────────┐
│ outcome  ┆ did_coef  ┆ p_event_clustered ┆ p_ticker_clustered ┆ n_obs  │
│ ---      ┆ ---       ┆

In [7]:
## Levels behind the DiD coefficients

# A coefficient alone doesn't show the underlying shares. Note the persistent baseline gap: retail sits at ~59.4% OTM vs procust ~54.1% even outside event windows.

for oc in ["otm", "otm_put", "lt_100"]:
    panel = build_diff_in_diff_panel(outcome=oc)
    levels = (
        panel.group_by(["participant_group", "is_near_event"])
        .agg(pl.col("share").mean().alias("mean_share"), pl.len().alias("n"))
        .sort("participant_group", "is_near_event")
    )
    print(f"--- {oc} ---")
    print(levels)
    print()

Matched 97,270 of 163,010 events to CBOE trading data
Panel has 317,190 rows (up to 2 periods x 2 groups per event)
--- otm ---
shape: (4, 4)
┌───────────────────┬───────────────┬────────────┬───────┐
│ participant_group ┆ is_near_event ┆ mean_share ┆ n     │
│ ---               ┆ ---           ┆ ---        ┆ ---   │
│ str               ┆ bool          ┆ f64        ┆ u32   │
╞═══════════════════╪═══════════════╪════════════╪═══════╡
│ procust           ┆ false         ┆ 0.541283   ┆ 74481 │
│ procust           ┆ true          ┆ 0.542961   ┆ 48325 │
│ retail            ┆ false         ┆ 0.594197   ┆ 97197 │
│ retail            ┆ true          ┆ 0.606598   ┆ 97187 │
└───────────────────┴───────────────┴────────────┴───────┘

Matched 97,270 of 163,010 events to CBOE trading data
Panel has 317,190 rows (up to 2 periods x 2 groups per event)
--- otm_put ---
shape: (4, 4)
┌───────────────────┬───────────────┬────────────┬───────┐
│ participant_group ┆ is_near_event ┆ mean_share ┆ n     │
│ -

In [8]:
## Moneyness decomposition: where does the OTM effect come from?

# The combined OTM effect decomposes almost entirely into puts:
#   otm_put  = +0.0097 (p < 0.0001)
#   otm_call = +0.0010 (p = 0.60, not significant)
#   otm      = +0.0107
# This runs against the lottery-preference literature's emphasis on OTM calls,
# and corroborates the separate call-share result (-0.0046) -- two independent
# measures pointing the same direction.
for oc in ["otm", "otm_call", "otm_put"]:
    panel = build_diff_in_diff_panel(outcome=oc)
    m = run_diff_in_diff(panel, cluster_by="ticker")
    print(f"{oc:9s} treat:post = {m.params['treat:post']:+.4f}, p = {m.pvalues['treat:post']:.4f}")

Matched 97,270 of 163,010 events to CBOE trading data
Panel has 317,190 rows (up to 2 periods x 2 groups per event)
otm       treat:post = +0.0107, p = 0.0000
Matched 97,270 of 163,010 events to CBOE trading data
Panel has 317,190 rows (up to 2 periods x 2 groups per event)
otm_call  treat:post = +0.0010, p = 0.6043
Matched 97,270 of 163,010 events to CBOE trading data
Panel has 317,190 rows (up to 2 periods x 2 groups per event)
otm_put   treat:post = +0.0097, p = 0.0000


In [9]:
## DV3: open vs. close positions (speculative intent)

from analysis.event_window_profile import build_diff_in_diff_panel, run_diff_in_diff
import polars as pl

panel_open = build_diff_in_diff_panel(outcome="open")

m1 = run_diff_in_diff(panel_open, cluster_by="event")
m2 = run_diff_in_diff(panel_open, cluster_by="ticker")
print(f"[open, event-clustered]  treat:post = {m1.params['treat:post']:+.4f}, p = {m1.pvalues['treat:post']:.4f}")
print(f"[open, ticker-clustered] treat:post = {m2.params['treat:post']:+.4f}, p = {m2.pvalues['treat:post']:.4f}")

print()
print(
    panel_open.group_by(["participant_group", "is_near_event"])
    .agg(pl.col("share").mean().alias("mean_open_share"), pl.len().alias("n"))
    .sort("participant_group", "is_near_event")
)

Matched 97,270 of 163,010 events to CBOE trading data
Panel has 317,190 rows (up to 2 periods x 2 groups per event)
[open, event-clustered]  treat:post = +0.0322, p = 0.0000
[open, ticker-clustered] treat:post = +0.0322, p = 0.0000

shape: (4, 4)
┌───────────────────┬───────────────┬─────────────────┬───────┐
│ participant_group ┆ is_near_event ┆ mean_open_share ┆ n     │
│ ---               ┆ ---           ┆ ---             ┆ ---   │
│ str               ┆ bool          ┆ f64             ┆ u32   │
╞═══════════════════╪═══════════════╪═════════════════╪═══════╡
│ procust           ┆ false         ┆ 0.771756        ┆ 74481 │
│ procust           ┆ true          ┆ 0.740267        ┆ 48325 │
│ retail            ┆ false         ┆ 0.651122        ┆ 97197 │
│ retail            ┆ true          ┆ 0.651809        ┆ 97187 │
└───────────────────┴───────────────┴─────────────────┴───────┘


In [10]:
## Mega cap divergence: volume weighted vs. equal weighted

# Ten tickers (AAPL, AMD, AMZN, BAC, C, FB, GE, MSF, NFLX, TSLA) behave oppositely to the rest of the universe on position size, and carry enough volume to flip the sign of any volume weighted aggregate. This is the empircal justifcation for the firm size control.

from analysis.event_window_profile import compare_weighting_schemes, compare_top_n_tickers

compare_weighting_schemes(outcome="lt_100")
print()
compare_top_n_tickers(outcome="lt_100", top_n=10)

Matched 97,270 of 163,010 events to CBOE trading data

=== Volume-weighted (every event-day counted by its own volume) ===
  retail: shift = +0.0142
  procust: shift = +0.0268

=== Equal-weighted per event (every firm-event counted once) ===
  retail: shift = +0.0758
  procust: shift = +0.0333

Top 10 tickers by volume: ['AAPL', 'AMD', 'AMZN', 'BAC', 'C', 'FB', 'GE', 'MSFT', 'NFLX', 'TSLA']

--- TOP 10 ---
  retail: shift = +0.0024
  procust: shift = +0.0529

--- EVERYONE ELSE ---
  retail: shift = +0.0761
  procust: shift = +0.0339



In [11]:
## Continious dispersion regresstion: RQ1 AND RQ2

from analysis.event_window_profile import build_diff_in_diff_panel, run_dispersion_regression

panel_otm = build_diff_in_diff_panel(outcome="otm")

print("=== Triple interaction: does retail's near-event shift grow with dispersion? ===")
m = run_dispersion_regression(panel_otm, spec="triple", cluster_by="ticker")
print(m.summary().tables[1])

print("\n=== Within earnings windows only ===")
m2 = run_dispersion_regression(panel_otm, spec="near_only", cluster_by="ticker")
print(m2.summary().tables[1])

Matched 97,270 of 163,010 events to CBOE trading data
Panel has 317,190 rows (up to 2 periods x 2 groups per event)
=== Triple interaction: does retail's near-event shift grow with dispersion? ===
                            coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept                 0.5414      0.002    258.380      0.000       0.537       0.545
dispersion                0.0073      0.002      4.240      0.000       0.004       0.011
treat                     0.0526      0.002     27.966      0.000       0.049       0.056
dispersion:treat          0.0015      0.002      0.940      0.347      -0.002       0.005
post                      0.0021      0.002      0.992      0.321      -0.002       0.006
dispersion:post           0.0012      0.002      0.561      0.575      -0.003       0.005
treat:post                0.0104      0.002      4.737      0.000       0.006      

In [12]:
## RQ1: does dispersion predict retail trading volume?

print("=== Log average daily volume ===")
mv = run_dispersion_regression(panel_otm, spec="triple", outcome_var="log_volume", cluster_by="ticker")
print(mv.summary().tables[1])

=== Log average daily volume ===
                            coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept                 0.9573      0.033     29.397      0.000       0.893       1.021
dispersion               -0.0895      0.017     -5.317      0.000      -0.122      -0.056
treat                     3.6819      0.013    279.642      0.000       3.656       3.708
dispersion:treat          0.1250      0.010     12.915      0.000       0.106       0.144
post                      1.1803      0.016     74.140      0.000       1.149       1.212
dispersion:post           0.0320      0.012      2.586      0.010       0.008       0.056
treat:post               -0.9573      0.018    -53.076      0.000      -0.993      -0.922
dispersion:treat:post    -0.0918      0.013     -6.956      0.000      -0.118      -0.066


In [13]:
## Is the dispersion effect actually a size effect?

from analysis.event_window_profile import (
    build_diff_in_diff_panel, run_dispersion_regression,
    add_market_cap, exclude_top_n_tickers,
)

panel_otm = build_diff_in_diff_panel(outcome="otm")

print("=== 1. Baseline (no size control) ===")
m1 = run_dispersion_regression(panel_otm, spec="triple", cluster_by="ticker")
print(f"dispersion:treat:post = {m1.params['dispersion:treat:post']:+.4f}, p = {m1.pvalues['dispersion:treat:post']:.4f}")

print("\n=== 2. Excluding the ten divergent mega-caps ===")
panel_ex = exclude_top_n_tickers(panel_otm, n=10)
m2 = run_dispersion_regression(panel_ex, spec="triple", cluster_by="ticker")
print(f"dispersion:treat:post = {m2.params['dispersion:treat:post']:+.4f}, p = {m2.pvalues['dispersion:treat:post']:.4f}")

print("\n=== 3. Controlling for market cap ===")
panel_mc = add_market_cap(panel_otm)
m3 = run_dispersion_regression(panel_mc, spec="triple", cluster_by="ticker", controls=["log_mktcap"])
print(m3.summary().tables[1])

Matched 97,270 of 163,010 events to CBOE trading data
Panel has 317,190 rows (up to 2 periods x 2 groups per event)
=== 1. Baseline (no size control) ===
dispersion:treat:post = -0.0046, p = 0.0379

=== 2. Excluding the ten divergent mega-caps ===
Excluding top 10 tickers by volume: ['AAPL', 'AMD', 'AMZN', 'BAC', 'C', 'FB', 'GE', 'MSFT', 'NFLX', 'TSLA']
  317,190 -> 315,465 panel rows
dispersion:treat:post = -0.0046, p = 0.0395

=== 3. Controlling for market cap ===
Market cap matched for 316,171 of 317,190 panel rows (99.7%)
  Dropped 1,019 rows missing control values (316,171 remain)
                            coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept                 0.5413      0.002    258.693      0.000       0.537       0.545
dispersion                0.0136      0.002      7.974      0.000       0.010       0.017
treat                     0.0529      0.002    

In [14]:
# Continuation of dispersion effect focusing on volume

print("=== RQ1 volume, with size control ===")
mv = run_dispersion_regression(panel_mc, spec="triple", outcome_var="log_volume",
                               cluster_by="ticker", controls=["log_mktcap"])
print(mv.summary().tables[1])

=== RQ1 volume, with size control ===
  Dropped 1,019 rows missing control values (316,171 remain)
                            coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept                 0.8913      0.023     38.204      0.000       0.846       0.937
dispersion                0.1170      0.014      8.454      0.000       0.090       0.144
treat                     3.8251      0.012    324.609      0.000       3.802       3.848
dispersion:treat          0.0729      0.009      7.728      0.000       0.054       0.091
post                      1.0185      0.013     75.854      0.000       0.992       1.045
dispersion:post          -0.0106      0.012     -0.852      0.394      -0.035       0.014
treat:post               -0.7812      0.015    -51.438      0.000      -0.811      -0.751
dispersion:treat:post    -0.0231      0.013     -1.787      0.074      -0.048       0.002
l

In [15]:
## Robustness: does the dispersion effect survive a firm size control?

from analysis.event_window_profile import dispersion_robustness_table
import polars as pl

print("=== Composition outcomes (share) ===")
tbl = dispersion_robustness_table(outcomes=["otm", "otm_put", "lt_100", "call", "open"])
with pl.Config(tbl_cols=12, tbl_width_chars=220, float_precision=4):
    print(tbl)

=== Composition outcomes (share) ===
shape: (5, 9)
┌─────────┬──────────┬────────┬──────────┬────────┬────────────┬─────────┬──────────┬────────┐
│ outcome ┆ did_base ┆ p_base ┆ did_ctrl ┆ p_ctrl ┆ cross_ctrl ┆ p_cross ┆ size_did ┆ p_size │
│ ---     ┆ ---      ┆ ---    ┆ ---      ┆ ---    ┆ ---        ┆ ---     ┆ ---      ┆ ---    │
│ str     ┆ f64      ┆ f64    ┆ f64      ┆ f64    ┆ f64        ┆ f64     ┆ f64      ┆ f64    │
╞═════════╪══════════╪════════╪══════════╪════════╪════════════╪═════════╪══════════╪════════╡
│ otm     ┆ -0.0046  ┆ 0.0379 ┆ -0.0024  ┆ 0.2886 ┆ -0.0070    ┆ 0.0000  ┆ 0.0120   ┆ 0.0000 │
│ otm_put ┆ -0.0045  ┆ 0.0146 ┆ -0.0025  ┆ 0.1781 ┆ -0.0014    ┆ 0.2813  ┆ 0.0116   ┆ 0.0000 │
│ lt_100  ┆ 0.0027   ┆ 0.0710 ┆ 0.0004   ┆ 0.8095 ┆ -0.0031    ┆ 0.0238  ┆ -0.0116  ┆ 0.0000 │
│ call    ┆ 0.0023   ┆ 0.2997 ┆ -0.0001  ┆ 0.9509 ┆ -0.0013    ┆ 0.3854  ┆ -0.0146  ┆ 0.0000 │
│ open    ┆ -0.0055  ┆ 0.0054 ┆ -0.0047  ┆ 0.0197 ┆ 0.0027     ┆ 0.0628  ┆ 0.0023   ┆ 0.2261 │

In [16]:
# continuation of dispersion robustness check

print("=== Volume outcome (RQ1) ===")
tbl_vol = dispersion_robustness_table(outcomes=["otm"], outcome_var="log_volume")
with pl.Config(tbl_cols=12, tbl_width_chars=220, float_precision=4):
    print(tbl_vol)

=== Volume outcome (RQ1) ===
shape: (1, 9)
┌─────────┬──────────┬────────┬──────────┬────────┬────────────┬─────────┬──────────┬────────┐
│ outcome ┆ did_base ┆ p_base ┆ did_ctrl ┆ p_ctrl ┆ cross_ctrl ┆ p_cross ┆ size_did ┆ p_size │
│ ---     ┆ ---      ┆ ---    ┆ ---      ┆ ---    ┆ ---        ┆ ---     ┆ ---      ┆ ---    │
│ str     ┆ f64      ┆ f64    ┆ f64      ┆ f64    ┆ f64        ┆ f64     ┆ f64      ┆ f64    │
╞═════════╪══════════╪════════╪══════════╪════════╪════════════╪═════════╪══════════╪════════╡
│ otm     ┆ -0.0918  ┆ 0.0000 ┆ -0.0231  ┆ 0.0740 ┆ 0.0729     ┆ 0.0000  ┆ 0.3879   ┆ 0.0000 │
└─────────┴──────────┴────────┴──────────┴────────┴────────────┴─────────┴──────────┴────────┘


In [17]:
panel_mc = add_market_cap(build_diff_in_diff_panel(outcome="otm"))
m = run_dispersion_regression(panel_mc, spec="near_only", cluster_by="ticker", controls=["log_mktcap"])
print(m.summary().tables[1])

Matched 97,270 of 163,010 events to CBOE trading data
Panel has 317,190 rows (up to 2 periods x 2 groups per event)
Market cap matched for 316,171 of 317,190 panel rows (99.7%)
  Dropped 460 rows missing control values (145,052 remain)
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept            0.5382      0.002    224.193      0.000       0.533       0.543
dispersion           0.0145      0.002      6.899      0.000       0.010       0.019
treat                0.0693      0.002     31.895      0.000       0.065       0.074
dispersion:treat    -0.0094      0.002     -4.542      0.000      -0.014      -0.005
log_mktcap           0.0268      0.002     11.258      0.000       0.022       0.032
log_mktcap:treat    -0.0347      0.002    -15.921      0.000      -0.039      -0.030


In [18]:
## Quartile breakdown

# establishes whether our convergence is smooth across the dispersion distribution or driven by one tail

panel_q = panel_mc.with_columns(
    pl.col("dispersion_scaled").qcut(4, labels=["Q1_low","Q2","Q3","Q4_high"]).alias("disp_q")
).filter(pl.col("is_near_event"))

print(
    panel_q.group_by(["disp_q", "participant_group"])
    .agg(pl.col("share").mean().alias("mean_otm_share"), pl.len().alias("n"))
    .sort("disp_q", "participant_group")
)

shape: (8, 4)
┌─────────┬───────────────────┬────────────────┬───────┐
│ disp_q  ┆ participant_group ┆ mean_otm_share ┆ n     │
│ ---     ┆ ---               ┆ ---            ┆ ---   │
│ cat     ┆ str               ┆ f64            ┆ u32   │
╞═════════╪═══════════════════╪════════════════╪═══════╡
│ Q1_low  ┆ procust           ┆ 0.543361       ┆ 13813 │
│ Q1_low  ┆ retail            ┆ 0.596625       ┆ 23515 │
│ Q2      ┆ procust           ┆ 0.533005       ┆ 12655 │
│ Q2      ┆ retail            ┆ 0.597192       ┆ 23926 │
│ Q3      ┆ procust           ┆ 0.536354       ┆ 11297 │
│ Q3      ┆ retail            ┆ 0.61118        ┆ 24537 │
│ Q4_high ┆ procust           ┆ 0.561439       ┆ 10560 │
│ Q4_high ┆ retail            ┆ 0.620367       ┆ 25209 │
└─────────┴───────────────────┴────────────────┴───────┘


In [19]:
# Coninuation of quartile breakdown adjusted to check for linearity once size is held constant

panel_q = panel_mc.with_columns(
    pl.col("dispersion_scaled").qcut(4, labels=["Q1_low","Q2","Q3","Q4_high"]).alias("disp_q")
)

m_q = run_dispersion_regression(
    panel_q.filter(pl.col("is_near_event")),
    spec="near_only", cluster_by="ticker", controls=["log_mktcap"]
)

# and the size profile across quartiles, to see how tightly the two are bound
print(
    panel_q.filter(pl.col("is_near_event"))
    .group_by("disp_q")
    .agg(
        pl.col("log_mktcap").mean().alias("mean_log_mktcap"),
        pl.col("dispersion_scaled").mean().alias("mean_dispersion"),
        pl.len().alias("n"),
    )
    .sort("disp_q")
)

  Dropped 460 rows missing control values (145,052 remain)
shape: (4, 4)
┌─────────┬─────────────────┬─────────────────┬───────┐
│ disp_q  ┆ mean_log_mktcap ┆ mean_dispersion ┆ n     │
│ ---     ┆ ---             ┆ ---             ┆ ---   │
│ cat     ┆ f64             ┆ f64             ┆ u32   │
╞═════════╪═════════════════╪═════════════════╪═══════╡
│ Q1_low  ┆ 15.94004        ┆ 0.019422        ┆ 37328 │
│ Q2      ┆ 15.371297       ┆ 0.057118        ┆ 36581 │
│ Q3      ┆ 14.690766       ┆ 0.136579        ┆ 35834 │
│ Q4_high ┆ 14.338974       ┆ 0.62787         ┆ 35769 │
└─────────┴─────────────────┴─────────────────┴───────┘


In [20]:
## Log transformation of the dispersion to fix skew.

# Allows for comparison of between two specification which tells us how much the result depends on functional form

# log-transformed dispersion (small constant so zeros survive the log)
panel_log = panel_mc.with_columns(
    (pl.col("dispersion_scaled") + 0.01).log().alias("dispersion_scaled")
)
m_log = run_dispersion_regression(
    panel_log, spec="near_only", cluster_by="ticker", controls=["log_mktcap"]
)
print("=== log(dispersion) ===")
print(m_log.summary().tables[1])

  Dropped 460 rows missing control values (145,052 remain)
=== log(dispersion) ===
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept            0.5381      0.002    224.194      0.000       0.533       0.543
dispersion           0.0155      0.002      7.116      0.000       0.011       0.020
treat                0.0694      0.002     31.993      0.000       0.065       0.074
dispersion:treat    -0.0082      0.002     -3.944      0.000      -0.012      -0.004
log_mktcap           0.0291      0.002     11.890      0.000       0.024       0.034
log_mktcap:treat    -0.0356      0.002    -15.935      0.000      -0.040      -0.031


In [21]:
# Continuation of log transofrmation, this version is assumption free, which lets the data choose its own shape

import statsmodels.formula.api as smf
import numpy as np

df = (
    panel_mc.filter(pl.col("is_near_event"))
    .with_columns(pl.col("dispersion_scaled").qcut(4, labels=["Q1","Q2","Q3","Q4"]).alias("disp_q"))
    .to_pandas()
    .dropna(subset=["log_mktcap"])
)
df["treat"] = (df["participant_group"] == "retail").astype(int)
df["log_mktcap_z"] = (df["log_mktcap"] - df["log_mktcap"].mean()) / df["log_mktcap"].std()

m_dum = smf.ols("share ~ C(disp_q) * treat + log_mktcap_z * treat", data=df).fit(
    cov_type="cluster", cov_kwds={"groups": df["resolved_ticker"]}
)
print(m_dum.summary().tables[1])

                            coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept                 0.5236      0.004    127.207      0.000       0.515       0.532
C(disp_q)[T.Q2]          -0.0002      0.005     -0.048      0.961      -0.010       0.010
C(disp_q)[T.Q3]           0.0134      0.006      2.408      0.016       0.002       0.024
C(disp_q)[T.Q4]           0.0449      0.006      7.630      0.000       0.033       0.056
treat                     0.0762      0.004     19.632      0.000       0.069       0.084
C(disp_q)[T.Q2]:treat    -0.0003      0.005     -0.060      0.952      -0.010       0.009
C(disp_q)[T.Q3]:treat    -0.0024      0.005     -0.454      0.650      -0.013       0.008
C(disp_q)[T.Q4]:treat    -0.0246      0.006     -4.409      0.000      -0.035      -0.014
log_mktcap_z              0.0296      0.002     12.070      0.000       0.025       0.034
log_mktcap

In [22]:
## Quartile specification on volume outcome

# panel_mc is outcome-independent for volume (avg_daily_vol = total_vol / n_days),
# so the panel built for "otm" works fine here
df_all = (
    panel_mc
    .filter(pl.col("avg_daily_vol") > 0)
    .with_columns(pl.col("dispersion_scaled").qcut(4, labels=["Q1","Q2","Q3","Q4"]).alias("disp_q"))
    .to_pandas()
    .dropna(subset=["log_mktcap"])
)
df_all["treat"] = (df_all["participant_group"] == "retail").astype(int)
df_all["log_mktcap_z"] = (df_all["log_mktcap"] - df_all["log_mktcap"].mean()) / df_all["log_mktcap"].std()
df_all["y"] = np.log(df_all["avg_daily_vol"])

for label, sub in [("NEAR-EVENT", df_all[df_all["is_near_event"]]),
                   ("BASELINE",   df_all[~df_all["is_near_event"]])]:
    m = smf.ols("y ~ C(disp_q) * treat + log_mktcap_z * treat", data=sub).fit(
        cov_type="cluster", cov_kwds={"groups": sub["resolved_ticker"]}
    )
    print(f"=== {label} (n = {int(m.nobs):,}) ===")
    print(m.summary().tables[1])
    print()

=== NEAR-EVENT (n = 145,052) ===
                            coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept                 1.7038      0.042     40.177      0.000       1.621       1.787
C(disp_q)[T.Q2]           0.0744      0.047      1.577      0.115      -0.018       0.167
C(disp_q)[T.Q3]           0.3029      0.054      5.596      0.000       0.197       0.409
C(disp_q)[T.Q4]           0.4410      0.061      7.266      0.000       0.322       0.560
treat                     2.9204      0.030     97.461      0.000       2.862       2.979
C(disp_q)[T.Q2]:treat     0.0701      0.032      2.177      0.029       0.007       0.133
C(disp_q)[T.Q3]:treat     0.1738      0.037      4.723      0.000       0.102       0.246
C(disp_q)[T.Q4]:treat     0.2664      0.042      6.273      0.000       0.183       0.350
log_mktcap_z              0.8116      0.034     24.067      0.000  

In [23]:
## Investigating the near event firm size reversal

from analysis.event_window_profile import build_diff_in_diff_panel, add_market_cap, investigate_size_reversal
import polars as pl

panel_mc = add_market_cap(build_diff_in_diff_panel(outcome="otm"))
res = investigate_size_reversal(panel_mc)

for name, tbl in res.items():
    print(f"=== {name} ===")
    with pl.Config(tbl_rows=20, float_precision=3):
        print(tbl)
    print()

Matched 97,270 of 163,010 events to CBOE trading data
Panel has 317,190 rows (up to 2 periods x 2 groups per event)
Market cap matched for 316,171 of 317,190 panel rows (99.7%)
=== selection ===
shape: (3, 3)
┌───────────────┬─────────────────┬────────┐
│ presence      ┆ mean_log_mktcap ┆ n      │
│ ---           ┆ ---             ┆ ---    │
│ str           ┆ f64             ┆ u32    │
╞═══════════════╪═════════════════╪════════╡
│ baseline_only ┆ 14.362          ┆ 28033  │
│ both          ┆ 15.110          ┆ 143086 │
│ near_only     ┆ 14.122          ┆ 1966   │
└───────────────┴─────────────────┴────────┘

=== by_size_quartile ===
shape: (16, 5)
┌──────────┬───────────────────┬───────────────┬──────────────┬───────┐
│ size_q   ┆ participant_group ┆ is_near_event ┆ mean_log_vol ┆ n     │
│ ---      ┆ ---               ┆ ---           ┆ ---          ┆ ---   │
│ cat      ┆ str               ┆ bool          ┆ f64          ┆ u32   │
╞══════════╪═══════════════════╪═══════════════╪═════════

In [24]:
# Contiuation of size reversal observation

import polars as pl
import statsmodels.formula.api as smf
import numpy as np

complete = (
    panel_mc.filter(pl.col("log_mktcap").is_not_null() & (pl.col("avg_daily_vol") > 0))
    .group_by(["resolved_ticker", "ANNDATS_ACT"])
    .agg(pl.len().alias("n_cells"))
    .filter(pl.col("n_cells") == 4)
    .select(["resolved_ticker", "ANNDATS_ACT"])
)
balanced = panel_mc.join(complete, on=["resolved_ticker", "ANNDATS_ACT"], how="inner")
print(f"Balanced panel: {balanced.height:,} rows (from {panel_mc.height:,})")

df = balanced.filter(pl.col("avg_daily_vol") > 0).to_pandas().dropna(subset=["log_mktcap"])
df["treat"] = (df["participant_group"] == "retail").astype(int)
df["post"] = df["is_near_event"].astype(int)
df["log_mktcap_z"] = (df["log_mktcap"] - df["log_mktcap"].mean()) / df["log_mktcap"].std()
df["y"] = np.log(df["avg_daily_vol"])

m = smf.ols("y ~ log_mktcap_z * treat * post", data=df).fit(
    cov_type="cluster", cov_kwds={"groups": df["resolved_ticker"]}
)
print(m.summary().tables[1])

Balanced panel: 184,868 rows (from 317,190)
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                   1.8286      0.027     66.766      0.000       1.775       1.882
log_mktcap_z                0.8933      0.030     29.783      0.000       0.835       0.952
treat                       3.7081      0.011    332.203      0.000       3.686       3.730
log_mktcap_z:treat         -0.1182      0.011    -10.995      0.000      -0.139      -0.097
post                        0.3761      0.010     36.287      0.000       0.356       0.396
log_mktcap_z:post          -0.1787      0.010    -17.418      0.000      -0.199      -0.159
treat:post                  0.1115      0.010     10.833      0.000       0.091       0.132
log_mktcap_z:treat:post     0.2024      0.010     20.259      0.000       0.183       0.222


In [25]:
## Re-running headline DID results on balanced panel

from analysis.event_window_profile import run_diff_in_diff

for oc in ["otm", "otm_put", "lt_100", "call", "open"]:
    p_full = build_diff_in_diff_panel(outcome=oc, verbose=False)
    p_bal = p_full.join(complete, on=["resolved_ticker", "ANNDATS_ACT"], how="inner")
    m_f = run_diff_in_diff(p_full, cluster_by="ticker")
    m_b = run_diff_in_diff(p_bal, cluster_by="ticker")
    print(f"{oc:9s} full = {m_f.params['treat:post']:+.4f} (p={m_f.pvalues['treat:post']:.4f})  "
          f"balanced = {m_b.params['treat:post']:+.4f} (p={m_b.pvalues['treat:post']:.4f})")

otm       full = +0.0107 (p=0.0000)  balanced = +0.0209 (p=0.0000)
otm_put   full = +0.0097 (p=0.0000)  balanced = +0.0182 (p=0.0000)
lt_100    full = +0.0425 (p=0.0000)  balanced = +0.0105 (p=0.0000)
call      full = -0.0046 (p=0.0321)  balanced = -0.0148 (p=0.0000)
open      full = +0.0322 (p=0.0000)  balanced = +0.0230 (p=0.0000)
